In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

In [0]:
%run /Workspace/Users/leoknaw@gmail.com/consolidated_pipeline/setup/utilities

In [0]:
print(bronze, silver, gold)
# If you want to check the new DataFrame, you can add:
# print(df_with_product_code.columns)

In [0]:
dbutils.widgets.text("catalog", "fmcg", "Catalog")
dbutils.widgets.text("data_source", "products", "Data Source")

catalog = dbutils.widgets.get("catalog")
data_source = dbutils.widgets.get("data_source")

base_path = f's3://atlikon/full_load/{data_source}'
landing_path = f'{base_path}/landing'
processed_path = f'{base_path}/processed'

print(base_path)
landing_path, processed_path

In [0]:
df = (
    spark.read.format("csv")
        .option("header", True)
        .option("inferSchema", True)
        .load(f'{landing_path}/*.csv')
        .withColumn("read_timestamp", F.current_timestamp())
        .select("*", "_metadata.file_name", "_metadata.file_size")
)

In [0]:
display(df)

In [0]:
df.write.format("delta")\
    .option("delta.enableChangeDataFeed", "true")\
    .mode("overwrite")\
    .saveAsTable(f"{catalog}.{bronze}.{data_source}")

In [0]:
files = dbutils.fs.ls(f"{landing_path}")

for file in files:
    dbutils.fs.mv(file.path, f"{processed_path}/{file.name}", True)

In [0]:
catalog = dbutils.widgets.get("catalog")
data_source = dbutils.widgets.get("data_source")

In [0]:
df_silver = spark.sql(f"SELECT * FROM {catalog}.{bronze}.{data_source}")
display(df_silver)

In [0]:
df_silver = df_silver.filter(F.col("order_qty").isNotNull())

In [0]:
df_orders = df_silver.withColumn("customer_id",
                                 F.when(F.col("customer_id").rlike(r"^[0-9]+$"), F.col("customer_id"))
                                    .otherwise("999999")
                                    .cast("string"))

display(df_orders)

In [0]:
df_orders.printSchema()

In [0]:
F.to_date("order_placement_date", "EEEE, MMMM dd, yyyy")

In [0]:
spark.conf.set("spark.sql.legacy.timeParserPolicy", "LEGACY")


In [0]:
df_orders = df_orders.withColumn(
    "order_placement_date",
    F.coalesce(
        F.try_to_date(F.trim("order_placement_date"), "EEEE, MMMM dd, yyyy"),
        F.try_to_date(F.trim("order_placement_date"), "dd/MM/yyyy"),
        F.try_to_date(F.trim("order_placement_date"), "dd-MM-yyyy"),
        F.try_to_date(F.trim("order_placement_date"), "yyyy-MM-dd"),
        F.try_to_date(F.trim("order_placement_date"), "yyyy/MM/dd")

    )
)

display(df_orders)

df_orders.printSchema()


In [0]:
df_products = spark.table("fmcg.gold.sb_dim_products")

df_joined = df_orders.join(df_products, on="product_id", how="inner").select(df_products["product_code"], df_orders["*"])

display(df_joined)

In [0]:
df_silver = df_joined

display(df_silver)

In [0]:
table_name = f"{catalog}.{silver}.{data_source}"

if not spark.catalog.tableExists(table_name):
    df_silver.write.format("delta").option("delta.enableChangeDataFeed", "true").mode("overwrite").saveAsTable(table_name)
else:
    delta_table = DeltaTable.forName(spark, table_name)
    delta_table.alias("silver").merge(
        df_silver.alias("bronze"),
        "silver.product_code = bronze.product_code AND silver.order_id = bronze.order_id AND silver.order_placement_date = bronze.order_placement_date AND silver.customer_id = bronze.customer_id"
    ).whenMatchedUpdateAll() \
     .whenNotMatchedInsertAll() \
     .execute()

In [0]:
df_gold = spark.sql("SELECT product_code, order_id, order_placement_date, customer_id as customer_code, product_id, order_qty from fmcg.silver.orders")
display(df_gold)

In [0]:
table_name = f"{catalog}.{gold}.{data_source}"

if not spark.catalog.tableExists(table_name):
    df_gold.write.format("delta").option("delta.enableChangeDataFeed", "true").mode("overwrite").saveAsTable(table_name)
else:
    delta_table = DeltaTable.forName(spark, table_name)
    delta_table.alias("gold").merge(
        df_gold.alias("bronze"),
        "gold.product_code = bronze.product_code AND gold.order_placement_date = bronze.order_placement_date AND gold.customer_id = bronze.customer_id"
    ).whenMatchedUpdateAll() \
     .whenNotMatchedInsertAll() \
     .execute()

In [0]:
df_child = spark.sql(f"SELECT * FROM fmcg.gold.{data_source}")
display(df_child)


In [0]:
df_monthly = df_child.withColumn("date", F.date_format("order_placement_date", "yyyy-MM-01")) \
    .groupBy("product_code", "date", "customer_id") \
    .agg(F.sum("order_qty").alias("sold_quantity")) \
    .select("product_code", "date", F.col("customer_id").alias("customer_code"), "sold_quantity")

display(df_monthly)

In [0]:
gold_parent_data = DeltaTable.forName(spark, f"{catalog}.{gold}.{data_source}")
gold_parent_data.alias("gold").merge(
    df_monthly.alias("new"),
    "gold.product_code = new.product_code AND gold.customer_code = new.customer_code AND gold.date = new.date AND gold.sold_quantity = new.sold_quantity",
).whenMatchedUpdateAll() \
.whenNotMatchedInsertAll() \
.execute()